# ColocVolume — usage example
> Individual-subject visualisation of SimNIBS e-fields via `simnibs_reader` + `ColocVolume`.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import simnibs_reader as snr
from visualizer import ColocVolume

## 0 — Paths (edit these)

In [ ]:
SIM_DIR = Path('/path/to/simulation_sub01_TDCS')
SEG_DIR = Path('/path/to/m2m_sub01')
LESION  = Path('/path/to/T1_brain_lesion.nii.gz')  # set to None to skip

ROI_COORDS = [28, -8, 54]
ROI_RADIUS = 5.0

## 1 — Load simulation + segmentation

In [ ]:
sim = snr.simulation(SIM_DIR)
seg = snr.segmentation(SEG_DIR)
sim.set_segmentation(seg)

print(sim)
print(sim.available_fields)

## 2 — Extract ROI and post-process

In [ ]:
efield = sim.magnE  # MNI space
# efield = sim.magnE_native  # subject space

roi = efield.get_roi(coords=ROI_COORDS, radius=ROI_RADIUS)
print(roi)
print(roi.stats())

In [ ]:
cleaned = roi.postprocess(
    smooth_fwhm    = 2.0,
    outlier_method = 'iqr',
    portion        = 0.95,
)
print(cleaned)
print(cleaned.stats())

## 3 — Build ColocVolume

In [ ]:
vol = ColocVolume()

vol.add_layer(
    seg.t1,
    role    = 'background',
    cmap    = 'gray',
    opacity = 0.15,
    label   = 'T1',
)

vol.add_layer(
    cleaned,          # ROIResult resolved via .mask_img
    role    = 'stat_map',
    cmap    = 'hot',
    opacity = 1.0,
    label   = 'magnE-cleaned',
    # clim  = (0.0, 0.15),  # uncomment to fix colour scale
)

if LESION is not None and LESION.exists():
    vol.add_layer(
        LESION,
        role    = 'overlay',
        color   = 'magenta',
        opacity = 0.4,
        label   = 'lesion',
    )

print(vol)

## 4a — 2-D ortho slices (nilearn)

In [ ]:
vol.plot_slices(
    cut_coords   = ROI_COORDS,
    display_mode = 'ortho',
    title        = f'magnE — ROI {ROI_COORDS}',
)

## 4b — 2-D parallel slices (nilearn)

In [ ]:
vol.plot_parallel_slices(axis='z', n_slices=10, title='magnE — axial overview')

In [ ]:
vol.plot_parallel_slices(axis='x', n_slices=8, title='magnE — sagittal overview')

## 5 — 3-D offscreen render (PyVista)

In [ ]:
frame = vol.plot_3d(camera_position='xy')
print(f'Rendered frame shape: {frame.shape}')

plt.figure(figsize=(6, 6))
plt.imshow(frame)
plt.axis('off')
plt.title('magnE — 3-D (xy)')
plt.tight_layout()
plt.show()

## 6 — Interactive 3-D (PyVista live window)

In [ ]:
# Blocks until the window is closed
vol.view_interactive(camera_position='xy', title='magnE — interactive')

## 7 — Tissue-filtered ROI (Gray-Matter only)

In [ ]:
# Requires sim.set_segmentation(seg) done above
roi_gm = (
    sim.magnE_native
       .get_roi(coords=ROI_COORDS, radius=10)
       .filter_tissue('Gray-Matter')
)
print(roi_gm)
print(roi_gm.stats())

In [ ]:
vol_gm = ColocVolume()
vol_gm.add_layer(seg.t1,  role='background', cmap='gray', opacity=0.15, label='T1')
vol_gm.add_layer(roi_gm,  role='stat_map',   cmap='hot',  opacity=1.0,  label='magnE-GM')

vol_gm.plot_slices(cut_coords=ROI_COORDS, title='magnE — Gray-Matter only')

## 8 — Cohort colour scale (cross-subject normalisation)
> Coming in next phase — use `vol.set_clim('magnE-cleaned', vmin, vmax)` with a pre-computed scale.

In [ ]:
# Example: fix scale from a pre-computed cohort range
# vol.set_clim('magnE-cleaned', vmin=0.0, vmax=0.15)
# vol.plot_slices(cut_coords=ROI_COORDS)